In [1]:
# bike_share_data_preparation.ipynb


import pandas as pd
import numpy as np
from zipfile import ZipFile
from pathlib import Path

BASE         = Path("/home/jovyan/AnalystAI")
RIDES_2024   = BASE / "data/metro_bike_share/2024"
STATIONS_DIR = BASE / "data/metro_bike_share/stations"
WEATHER_DIR  = BASE / "data/metro_bike_share/weather"

print("Loading all 4 quarters...")
parts = []
for q in range(1, 5):
    zip_path = RIDES_2024 / f"metro-trips-2024-q{q}.zip"
    with ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith('.csv'):
                df = pd.read_csv(zf.open(name), low_memory=False)
                parts.append(df)
                print(f"  Q{q}: {len(df):,} rows")

trips = pd.concat(parts, ignore_index=True)

print(f"\nTotal trips : {len(trips):,}")
print(f"   Columns     : {trips.columns.tolist()}")
print(f"   Shape       : {trips.shape}")
print(f"\nSample data:")
print(trips.head(3))

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


Loading all 4 quarters...
  Q1: 119,849 rows
  Q2: 143,616 rows
  Q3: 134,918 rows
  Q4: 120,854 rows

Total trips : 519,237
   Columns     : ['trip_id', 'duration', 'start_time', 'end_time', 'start_station', 'start_lat', 'start_lon', 'end_station', 'end_lat', 'end_lon', 'bike_id', 'plan_duration', 'trip_route_category', 'passholder_type', 'bike_type']
   Shape       : (519237, 15)

Sample data:
     trip_id  duration     start_time       end_time  start_station  \
0  341828178        87  1/1/2024 0:10  1/1/2024 1:37           4515   
1  341816845         6  1/1/2024 0:12  1/1/2024 0:18           4602   
2  341817147        20  1/1/2024 0:15  1/1/2024 0:35           3064   

   start_lat   start_lon  end_station    end_lat     end_lon  bike_id  \
0  34.039742 -118.442268         4564  34.035351 -118.434143    24169   
1  34.164951 -118.363632         4603  34.152142 -118.361954    15430   
2  34.046131 -118.257591         3081  34.031891 -118.250183     5913   

   plan_duration trip_r

In [3]:

# Load stations (with different encodings)
for encoding in ['utf-8', 'latin-1', 'cp1252']:
    try:
        stations = pd.read_csv(
            STATIONS_DIR / "metro-bike-share-stations-2025-07-01-1.csv",
            encoding=encoding
        )
        print(f"Stations loaded with encoding: {encoding}")
        break
    except UnicodeDecodeError:
        continue

stations.columns = stations.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True)

print("Stations")
print("  Shape  :", stations.shape)
print("  Columns:", stations.columns.tolist())
print(stations.head(3))

# Load weather
w1 = pd.read_csv(WEATHER_DIR / "meteostat_la_2024.csv")
w2 = pd.read_csv(WEATHER_DIR / "meteostat_la_2025.csv")
weather = pd.concat([w1, w2], ignore_index=True)
weather.columns = weather.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True)

print("\nWeather")
print("  Shape  :", weather.shape)
print("  Columns:", weather.columns.tolist())
print(weather.head(3))

Stations loaded with encoding: latin-1
Stations
  Shape  : (453, 7)
  Columns: ['kiosk_id', 'kiosk_name', 'go_live_date', 'region', 'status', 'latitude', 'longitude']
   kiosk_id       kiosk_name go_live_date region  status  latitude   longitude
0    3000.0  Virtual Station     7/7/2016    NaN  Active   0.00000    0.000000
1    3005.0     7th & Flower     7/7/2016   DTLA  Active  34.04850 -118.258537
2    3006.0      Olive & 8th     7/7/2016   DTLA  Active  34.04554 -118.256668

Weather
  Shape  : (16115, 12)
  Columns: ['time', 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']
                  time  temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt  \
0  2024-01-01 00:00:00  15.0  11.2  78.0   0.0   NaN   70.0  13.0   NaN   
1  2024-01-01 01:00:00  15.0  10.0  72.0   0.0   NaN  200.0   9.4   NaN   
2  2024-01-01 02:00:00  14.4  10.6  78.0   0.0   NaN  220.0   7.6   NaN   

     pres  tsun  coco  
0  1017.2   NaN   3.0  
1  1017.8   NaN   3.0  
2  1

In [4]:
# Clean trips, stations and weather

# --- Clean Trips ---
print("Cleaning trips...")
n0 = len(trips)

# Standardize column names
trips.columns = trips.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True)

# Parse datetime
trips['start_time'] = pd.to_datetime(trips['start_time'], errors='coerce')
trips['end_time']   = pd.to_datetime(trips['end_time'],   errors='coerce')

# Drop missing timestamps
trips = trips.dropna(subset=['start_time', 'end_time'])

# Keep only 2024
trips = trips[trips['start_time'].dt.year == 2024]

# Drop end before start
trips = trips[trips['end_time'] >= trips['start_time']]

# Drop duplicates
trips = trips.drop_duplicates(subset=['trip_id'])

# Drop virtual stations (lat=0, lon=0)
trips = trips[~((trips['start_lat'] == 0) & (trips['start_lon'] == 0))]
trips = trips[~((trips['end_lat']   == 0) & (trips['end_lon']   == 0))]

# Rename duration to duration_minutes
trips = trips.rename(columns={'duration': 'duration_minutes'})

# Standardize categoricals
trips['bike_type']       = trips['bike_type'].str.strip().str.title()
trips['passholder_type'] = trips['passholder_type'].str.strip().str.title()
trips['trip_route_category'] = trips['trip_route_category'].str.strip().str.title()

# Clip duration
trips['duration_minutes'] = trips['duration_minutes'].clip(lower=1, upper=1440)

trips = trips.reset_index(drop=True)
print(f"  Rows before : {n0:,}")
print(f"  Rows after  : {len(trips):,}")
print(f"  Dropped     : {n0 - len(trips):,}")
print(f"  Columns     : {trips.columns.tolist()}")

# --- Clean Stations ---
print("\nCleaning stations...")
n0 = len(stations)

# Rename columns to standard names
stations = stations.rename(columns={
    'kiosk_id'   : 'station_id',
    'kiosk_name' : 'station_name',
    'latitude'   : 'lat',
    'longitude'  : 'lon',
})

# Drop virtual station
stations = stations[~((stations['lat'] == 0) & (stations['lon'] == 0))]

# Drop missing coords
stations = stations.dropna(subset=['lat', 'lon'])

# Keep only active stations
stations = stations[stations['status'] == 'Active']

# Clean station_id
stations['station_id'] = stations['station_id'].astype(int).astype(str)

stations = stations.reset_index(drop=True)
print(f"  Rows before : {n0:,}")
print(f"  Rows after  : {len(stations):,}")
print(f"  Columns     : {stations.columns.tolist()}")

# --- Clean Weather ---
print("\nCleaning weather...")
n0 = len(weather)

# Rename columns to standard names
weather = weather.rename(columns={
    'time' : 'timestamp',
    'temp' : 'temp_c',
    'rhum' : 'rel_humidity',
    'prcp' : 'precip_mm',
    'wspd' : 'wind_speed_mps',
    'pres' : 'pressure_hpa',
    'coco' : 'weather_code',
})

# Parse timestamp and convert to LA timezone
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
weather['timestamp'] = weather['timestamp'].dt.tz_convert('America/Los_Angeles')

# Keep only 2024
weather = weather[weather['timestamp'].dt.year == 2024]

# Keep only needed columns
weather = weather[['timestamp', 'temp_c', 'rel_humidity',
                   'precip_mm', 'wind_speed_mps', 'pressure_hpa', 'weather_code']]

# Fill missing precip with 0
weather['precip_mm'] = weather['precip_mm'].fillna(0.0)

# Forward fill other columns
for col in ['temp_c', 'rel_humidity', 'wind_speed_mps', 'pressure_hpa', 'weather_code']:
    weather[col] = weather[col].ffill()

weather = weather.drop_duplicates(subset=['timestamp'])
weather = weather.reset_index(drop=True)

print(f"  Rows before : {n0:,}")
print(f"  Rows after  : {len(weather):,}")
print(f"  Columns     : {weather.columns.tolist()}")

print("\nAll three datasets cleaned!")

Cleaning trips...
  Rows before : 519,237
  Rows after  : 519,235
  Dropped     : 2
  Columns     : ['trip_id', 'duration_minutes', 'start_time', 'end_time', 'start_station', 'start_lat', 'start_lon', 'end_station', 'end_lat', 'end_lon', 'bike_id', 'plan_duration', 'trip_route_category', 'passholder_type', 'bike_type']

Cleaning stations...
  Rows before : 453
  Rows after  : 226
  Columns     : ['station_id', 'station_name', 'go_live_date', 'region', 'status', 'lat', 'lon']

Cleaning weather...
  Rows before : 16,115
  Rows after  : 8,784
  Columns     : ['timestamp', 'temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_mps', 'pressure_hpa', 'weather_code']

All three datasets cleaned!


In [5]:
# Building SQLite database

import sqlite3
from pathlib import Path

DB_PATH = Path("/home/jovyan/AnalystAI/database/bikeshare_2024.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# Remove old database if exists
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)

# --- Save trips ---
print("Saving trips table...")
trips_db = trips[[
    'trip_id', 'start_time', 'end_time', 'duration_minutes',
    'start_station', 'start_lat', 'start_lon',
    'end_station', 'end_lat', 'end_lon',
    'bike_id', 'bike_type', 'passholder_type', 'trip_route_category'
]].copy()

# Convert timestamps to string for SQLite
trips_db['start_time'] = trips_db['start_time'].dt.strftime('%Y-%m-%d %H:%M:%S')
trips_db['end_time']   = trips_db['end_time'].dt.strftime('%Y-%m-%d %H:%M:%S')

trips_db.to_sql('trips', conn, if_exists='replace', index=False)
print(f"  Rows saved : {len(trips_db):,}")

# --- Save stations ---
print("Saving stations table...")
stations_db = stations[['station_id', 'station_name', 'region', 'lat', 'lon']].copy()
stations_db.to_sql('stations', conn, if_exists='replace', index=False)
print(f"  Rows saved : {len(stations_db):,}")

# --- Save weather ---
print("Saving weather table...")
weather_db = weather.copy()
weather_db['timestamp'] = weather_db['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
weather_db.to_sql('weather', conn, if_exists='replace', index=False)
print(f"  Rows saved : {len(weather_db):,}")

conn.close()

# Verify
print("\nVerifying database...")
conn = sqlite3.connect(DB_PATH)
for table in ['trips', 'stations', 'weather']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    cols  = [r[1] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()]
    print(f"  {table:10} : {count:,} rows | columns: {cols}")
conn.close()

print(f"\nDatabase saved at: {DB_PATH}")
print(f"Database size    : {DB_PATH.stat().st_size / 1024 / 1024:.1f} MB")

Saving trips table...
  Rows saved : 519,235
Saving stations table...
  Rows saved : 226
Saving weather table...
  Rows saved : 8,784

Verifying database...
  trips      : 519,235 rows | columns: ['trip_id', 'start_time', 'end_time', 'duration_minutes', 'start_station', 'start_lat', 'start_lon', 'end_station', 'end_lat', 'end_lon', 'bike_id', 'bike_type', 'passholder_type', 'trip_route_category']
  stations   : 226 rows | columns: ['station_id', 'station_name', 'region', 'lat', 'lon']
  weather    : 8,784 rows | columns: ['timestamp', 'temp_c', 'rel_humidity', 'precip_mm', 'wind_speed_mps', 'pressure_hpa', 'weather_code']

Database saved at: /home/jovyan/AnalystAI/database/bikeshare_2024.sqlite
Database size    : 65.6 MB


In [6]:
# Upload to HuggingFace

from huggingface_hub import HfApi, login
from pathlib import Path

HF_TOKEN  = "Your API Key"
HF_REPO   = "Krupa1420/la-metro-bikeshare-2024"
DB_PATH   = Path("/home/jovyan/AnalystAI/database/bikeshare_2024.sqlite")

login(token=HF_TOKEN)
api = HfApi()

# Create dataset repo
try:
    api.create_repo(
        repo_id = HF_REPO,
        repo_type = "dataset",
        private = False,
        token = HF_TOKEN,
        exist_ok = True,
    )
    print(f"Repo ready: {HF_REPO}")
except Exception as e:
    print(f"Repo already exists: {e}")

# Create README
readme = """# LA Metro Bike-Share 2024

Real LA Metro Bike-Share data for 2024.
Clean and ready to use.

## Contents
- `bikeshare_2024.sqlite` - SQLite database with 3 tables

## Tables

### trips (519,235 rows)
| Column | Description |
|--------|-------------|
| trip_id | Unique trip identifier |
| start_time | Trip start timestamp |
| end_time | Trip end timestamp |
| duration_minutes | Trip duration in minutes |
| start_station | Start station ID |
| end_station | End station ID |
| bike_type | Electric or Standard |
| passholder_type | Monthly Pass, Walk-up, Flex Pass |
| trip_route_category | One Way or Round Trip |

### stations (226 rows)
| Column | Description |
|--------|-------------|
| station_id | Unique station identifier |
| station_name | Station name |
| region | Area (DTLA, Venice, etc.) |
| lat | Latitude |
| lon | Longitude |

### weather (8,784 rows)
| Column | Description |
|--------|-------------|
| timestamp | Hourly timestamp |
| temp_c | Temperature in Celsius |
| rel_humidity | Relative humidity % |
| precip_mm | Precipitation in mm |
| wind_speed_mps | Wind speed m/s |
| pressure_hpa | Pressure in hPa |
| weather_code | Weather condition code |

## Usage
```python
import sqlite3
import pandas as pd
from huggingface_hub import hf_hub_download

db_path = hf_hub_download(
    repo_id   = "Krupa1420/la-metro-bikeshare-2024",
    filename  = "bikeshare_2024.sqlite",
    repo_type = "dataset"
)

conn = sqlite3.connect(db_path)
df = pd.read_sql("SELECT * FROM trips LIMIT 5", conn)
print(df)
```

## Source
LA Metro Bike Share Open Data
"""

with open("/home/jovyan/AnalystAI/database/README.md", "w") as f:
    f.write(readme)

# Upload files
print("Uploading SQLite database (65 MB)...")
api.upload_file(
    path_or_fileobj = str(DB_PATH),
    path_in_repo    = "bikeshare_2024.sqlite",
    repo_id         = HF_REPO,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("SQLite uploaded!")

print("Uploading README...")
api.upload_file(
    path_or_fileobj = "/home/jovyan/AnalystAI/database/README.md",
    path_in_repo    = "README.md",
    repo_id         = HF_REPO,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("README uploaded!")

# Also save clean CSVs and upload
print("\nSaving and uploading CSVs...")
csv_dir = Path("/home/jovyan/AnalystAI/database")

# trips CSV
trips_db.to_csv(csv_dir / "trips.csv", index=False)
api.upload_file(
    path_or_fileobj = str(csv_dir / "trips.csv"),
    path_in_repo    = "trips.csv",
    repo_id         = HF_REPO,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("  trips.csv uploaded!")

# stations CSV
stations_db.to_csv(csv_dir / "stations.csv", index=False)
api.upload_file(
    path_or_fileobj = str(csv_dir / "stations.csv"),
    path_in_repo    = "stations.csv",
    repo_id         = HF_REPO,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("  stations.csv uploaded!")

# weather CSV
weather_db.to_csv(csv_dir / "weather.csv", index=False)
api.upload_file(
    path_or_fileobj = str(csv_dir / "weather.csv"),
    path_in_repo    = "weather.csv",
    repo_id         = HF_REPO,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("  weather.csv uploaded!")

print(f"\nDataset live at: https://huggingface.co/datasets/{HF_REPO}")

Repo ready: Krupa1420/la-metro-bikeshare-2024
Uploading SQLite database (65 MB)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

SQLite uploaded!
Uploading README...


/opt/conda/lib/python3.11/site-packages/huggingface_hub/hf_api.py:10227: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


README uploaded!

Saving and uploading CSVs...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  trips.csv uploaded!
  stations.csv uploaded!
  weather.csv uploaded!

Dataset live at: https://huggingface.co/datasets/Krupa1420/la-metro-bikeshare-2024
